In [10]:
!pip install pymupdf ollama

## Part C : Local Inference with Qwen MoE Models using Ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd
!apt install pciutils lshw

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 94 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,157 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 130169 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
Reading package lists... Done
Bu

In [5]:
import subprocess
import time
from urllib.request import urlopen

ollama_log = open("/tmp/ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Wait until the API is ready
for _ in range(30):
    try:
        print(urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read())

{"models":[]}


In [6]:
for model in ["qwen2.5vl:3b", "qwen2.5vl:7b", "qwen2.5vl:32b"]:
    with open("/kaggle/working/ollama-pull.log", "a") as log:
        subprocess.run(
            ["ollama", "pull", model],
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
    print(f"Downloaded {model}")

Downloaded qwen2.5vl:3b
Downloaded qwen2.5vl:7b
Downloaded qwen2.5vl:32b


In [16]:
import glob, json, time
from pathlib import Path
import pymupdf
from ollama import Client

MODELS = ["qwen2.5vl:3b", "qwen2.5vl:7b", "qwen2.5vl:32b"]
client = Client(host="http://localhost:11434", timeout=900)

pdf_path = Path("/kaggle/input/datasets/redaabanouas/reports/bilan_group_7.pdf")

PROMPT = """\
Tu es chargé d'extraire fidèlement les informations comptables d'un document.

Lis toutes les pages du PDF. Repère le bilan actif et le bilan passif, y compris
lorsqu'un tableau se poursuit sur plusieurs pages. Ignore les annexes qui ne
contiennent pas de postes du bilan.

Retourne uniquement un objet JSON valide, sans Markdown ni commentaire, conforme
exactement au schéma ci-dessous. Respecte l'ordre des clés montré dans chaque
objet. N'ajoute aucune clé et ne modifie pas les noms des clés.

{
  "document_type": "bilan",
  "period": {
    "start_date": null,
    "end_date": null
  },
  "assets": {
    "items": [
      {
        "label": "",
        "code": "",
        "gross": null,
        "depreciation_and_provisions": null,
        "net_current": null,
        "net_previous": null
      }
    ],
    "totals": {
      "fixed_assets": null,
      "current_assets": null,
      "cash": null,
      "total_assets": null
    }
  },
  "liabilities_and_equity": {
    "items": [
      {
        "label": "",
        "code": "",
        "current": null,
        "previous": null
      }
    ],
    "totals": {
      "equity": null,
      "long_term_funding": null,
      "current_liabilities": null,
      "cash_liabilities": null,
      "total_liabilities_and_equity": null
    }
  }
}

Règles d'extraction :

- Extrais toutes les rubriques principales de premier niveau portant des
  montants. Ignore les sous-rubriques indentées, les annexes et les lignes sans
  montant. Ne somme pas les sous-rubriques.
- Crée un objet par rubrique principale dans "items". Reproduis son libellé
  d'origine dans "label" et utilise un code anglais stable en snake_case.
- Pour chaque ligne de l'actif, lis les colonnes dans cet ordre :
  "Brut exercice", "Amortissements et provisions exercice", "Net exercice",
  "Net exercice précédent". Mappe-les respectivement à "gross",
  "depreciation_and_provisions", "net_current" et "net_previous".
- Pour chaque ligne du passif, lis les colonnes de l'exercice courant et de
  l'exercice précédent. Mappe-les respectivement à "current" et "previous".
- Chaque objet d'actif contient exactement, dans cet ordre, les clés "label",
  "code", "gross", "depreciation_and_provisions", "net_current" et
  "net_previous". Chaque objet de passif contient exactement, dans cet ordre,
  les clés "label", "code", "current" et "previous".
- Codes canoniques pour l'actif : non_value_assets, intangible_assets,
  tangible_assets, financial_assets, currency_translation_fixed_assets,
  inventory, receivables, marketable_securities,
  currency_translation_current_assets, cash.
- Codes canoniques pour le passif : equity, assimilated_equity,
  financing_debts, risk_provisions, currency_translation_liabilities,
  current_liabilities, cash_liabilities. Pour tout autre poste, crée un code
  anglais descriptif et stable en snake_case.
- Écris les montants comme des chaînes décimales, avec "." comme séparateur
  décimal et sans séparateur de milliers. Écris les dates au format YYYY-MM-DD.
- Utilise null si une valeur est absente, illisible ou ambiguë. Ne calcule et ne
  déduis aucune valeur.
- Pour les totaux de l'actif, reprends les montants nets imprimés des totaux I,
  II, III et I+II+III. Pour les totaux du passif, reprends les montants affichés.
  Place les totaux uniquement dans "totals", jamais dans "items".
- Si une rubrique de bilan est absente, retourne "items": [] et des totaux null
  pour cette partie.
- Réponds uniquement avec un JSON complet et valide conforme au schéma, sans
  commentaire ni clé supplémentaire.
"""

latencies = {}
RENDER_SCALE = 1.5

for model in MODELS:
    out = Path("/kaggle/working/out")
    out.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    with pymupdf.open(pdf_path) as pdf:
        imgs = [
            page.get_pixmap(matrix=pymupdf.Matrix(RENDER_SCALE, RENDER_SCALE), alpha=False).tobytes("png")
            for page in pdf
        ]

    r = client.chat(
        model=model,
        messages=[{"role": "user", "content": PROMPT, "images": imgs}],
        format="json",
        options={
            "num_ctx": 32768,
            "num_predict": 6000,
            "temperature": 0,
        },
    )

    raw = r.message.content
    elapsed = round(time.time() - t0, 3)
    
    latencies[model] = elapsed

    try:
        result = json.loads(r.message.content)
        result["model"] = model
        (out / f"{pdf_path.stem}_{model.replace(':', '_')}.json").write_text(
            json.dumps(result, ensure_ascii=False, indent=2)
        )
    except json.JSONDecodeError:
        (out / f"{pdf_path.stem}_{model.replace(':', '_')}.raw.txt").write_text(
            r.message.content
        )
        print("INVALID JSON:", model, "| done_reason:", r.done_reason)

    print(model, f"inference time : {elapsed} seconds")

latency_path = Path("/kaggle/working/out") / "latency.json"
latency_path.parent.mkdir(parents=True, exist_ok=True)
latency_path.write_text(json.dumps(latencies, indent=2))

qwen2.5vl:3b inference time : 183.658 seconds
qwen2.5vl:7b inference time : 59.321 seconds
qwen2.5vl:32b inference time : 519.356 seconds


83